# Unified OCR: แบ่งเขต + บัญชีรายชื่อ

ไฟล์นี้รวม OCR ทั้ง 2 ประเภทไว้ใน notebook เดียว:
- ไฟล์ไม่มี `บช` = แบ่งเขต
- ไฟล์มี `บช` / `บัญชีรายชื่อ` / `partylist` = บัญชีรายชื่อ

Output หลัก:
- `/content/constituency_results_gemini.csv`
- `/content/partylist_results_gemini.csv`

โค้ด parse ทีละหน้าเพื่อลดปัญหา JSON พัง และมีไฟล์ error/debug แยกให้ตรวจสอบ


In [ ]:
# ============================================================
# Install libraries
# ============================================================

!pip -q install google-api-python-client google-auth-httplib2 google-auth-oauthlib \
                 google-genai pandas pymupdf pillow tqdm rapidfuzz
!apt-get -qq update
!apt-get -qq install -y poppler-utils


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 137.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 169.7 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package poppler-utils.
(Reading database ... 122402 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
# ============================================================
# Unified OCR: แบ่งเขต + บัญชีรายชื่อ
# Output:
# 1) /content/constituency_results_gemini.csv
# 2) /content/partylist_results_gemini.csv
# ============================================================

import io
import os
import re
import json
import ast
import time
import math
import random
from pathlib import Path
from collections import deque

import pandas as pd
from tqdm.auto import tqdm
import fitz

from rapidfuzz import process, fuzz

from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

from google import genai
from google.genai import types


# ============================================================
# 1) Config
# ============================================================

FOLDER_URL = "https://drive.google.com/drive/folders/170C8kzr9J6VzdykufsLGWAckjtdHeoxA?usp=sharing"

# ใส่ API key ด้วยวิธีนี้ก่อนรัน:
# import os
os.environ["GEMINI_API_KEY"] =  ""
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")

# หรือใส่ตรงนี้แทน แต่อย่าแชร์ notebook ที่มี key จริง
# GEMINI_API_KEY = "YOUR_API_KEY"

OCR_MODEL = "gemini-2.5-flash"
PARSER_MODEL = "gemini-2.5-flash"

TARGET_PATTERN = "5ทับ18"
STRICT_TARGET_PATTERN = True

# ถ้าจะ test แค่บางไฟล์ ให้ใส่เลข เช่น 5
# ถ้าจะรันทั้งหมดให้เป็น None
MAX_FILES_TO_PROCESS = None

DEBUG_FILE_DETECTION = True

PDF_ZOOM = 2.4
REQUEST_DELAY_SECONDS = 1.0
MAX_RETRIES = 6
INITIAL_BACKOFF_SECONDS = 5.0
BACKOFF_MULTIPLIER = 2.0
MAX_BACKOFF_SECONDS = 90.0

BASE_DIR = Path("/content/unified_election_ocr_work")
DOWNLOAD_DIR = BASE_DIR / "downloads"
OCR_DIR = BASE_DIR / "ocr"
OUTPUT_DIR = Path("/content")

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
OCR_DIR.mkdir(parents=True, exist_ok=True)

CONSTITUENCY_OUTPUT = OUTPUT_DIR / "constituency_results_gemini.csv"
PARTYLIST_OUTPUT = OUTPUT_DIR / "partylist_results_gemini.csv"
SUMMARY_OUTPUT = OUTPUT_DIR / "all_summary_gemini.csv"
ERROR_OUTPUT = OUTPUT_DIR / "errors_gemini.csv"
DETECTION_DEBUG_OUTPUT = OUTPUT_DIR / "file_detection_debug.csv"


# ============================================================
# 2) Auth
# ============================================================

if not GEMINI_API_KEY:
    raise ValueError(
        "Please set GEMINI_API_KEY first. Example:\n"
        "import os\n"
        "os.environ['GEMINI_API_KEY'] = 'YOUR_API_KEY'"
    )

auth.authenticate_user()
drive_service = build("drive", "v3")
client = genai.Client(api_key=GEMINI_API_KEY)

print("Google Drive auth completed.")
print("Gemini client created.")
print("OCR_MODEL =", OCR_MODEL)
print("PARSER_MODEL =", PARSER_MODEL)


# ============================================================
# 3) Known location lexicon
# ============================================================

KNOWN_AMPHOE = [
    "อำเภอฝาง",
    "อำเภอแม่อาย",
    "อำเภอไชยปราการ",
    "ทต.แม่อาย",
]

KNOWN_TAMBON = [
    "ตำบลมะลิกา",
    "ตำบลแม่อาย",
    "ตำบลปงตำ",
    "ตำบลแม่ทะลบ",
    "ตำบลโป่งน้ำร้อน",
    "ตำบลม่อนปิ่น",
    "ตำบลแม่ข่า",
    "ตำบลแม่คะ",
    "ตำบลแม่งอน",
    "ตำบลแม่สูน",
    "ตำบลเวียง",
    "ตำบลสันทราย",
    "ทต.บ้านแม่ข่า",
    "ทต.เวียงฝาง",
    "ตำบลท่าตอน",
    "ตำบลบ้านหลวง",
    "ตำบลแม่นาวาง",
    "ตำบลแม่สาว",
    "ตำบลสันต้นหมื้อ",
]

LOCATION_LEXICON = {
    "อำเภอ/เขต": KNOWN_AMPHOE,
    "ตำบล/แขวง/เทศบาล": KNOWN_TAMBON,
}

FUZZY_SCORE_CUTOFF = 88


# ============================================================
# 4) Filename detection
# ============================================================

def extract_folder_id(url: str) -> str:
    m = re.search(r"/folders/([a-zA-Z0-9_-]+)", str(url))
    if not m:
        raise ValueError("หา folder id จากลิงก์ไม่ได้")
    return m.group(1)


ROOT_FOLDER_ID = extract_folder_id(FOLDER_URL)
print("ROOT_FOLDER_ID =", ROOT_FOLDER_ID)


def normalize_stem(name: str) -> str:
    name = str(name).strip().lower()
    name = re.sub(r"\s+", "", name)
    name = name.replace(".", "")
    name = name.replace("_", "ทับ")
    name = name.replace("-", "ทับ")
    name = name.replace("/", "ทับ")
    name = name.replace("\\", "ทับ")
    return name


def has_bch_marker(name: str) -> bool:
    raw = str(name)
    compact = re.sub(r"[\s\.()（）\-_]+", "", raw.lower())

    return (
        "บช" in compact
        or "บัญชีรายชื่อ" in compact
        or "partylist" in compact
        or "party-list" in raw.lower()
    )


def detect_document_type(filename_or_path: str) -> str:
    if has_bch_marker(filename_or_path):
        return "บัญชีรายชื่อ"
    return "แบ่งเขต"


def is_pdf_file(filename: str, mime_type: str = None) -> bool:
    filename = str(filename).lower()
    mime_type = str(mime_type or "").lower()
    return filename.endswith(".pdf") or mime_type == "application/pdf"


def is_target_pdf_name(filename: str, mime_type: str = None) -> bool:
    if not is_pdf_file(filename, mime_type):
        return False

    norm = normalize_stem(filename)

    if STRICT_TARGET_PATTERN and TARGET_PATTERN not in norm:
        return False

    has_ss = "สส" in norm
    return has_ss


def explain_detection(filename: str, mime_type: str = None) -> str:
    checks = {
        "is_pdf": is_pdf_file(filename, mime_type),
        "has_target_5ทับ18": TARGET_PATTERN in normalize_stem(filename),
        "has_ss": "สส" in normalize_stem(filename),
        "has_bch": has_bch_marker(filename),
        "document_type": detect_document_type(filename),
        "selected": is_target_pdf_name(filename, mime_type),
    }
    return " | ".join([f"{k}={v}" for k, v in checks.items()])


def safe_filename(name: str) -> str:
    return re.sub(r'[\\/:*?"<>|]+', "_", str(name).strip())


# ============================================================
# 5) Google Drive list + download
# ============================================================

def list_children(folder_id):
    results = []
    page_token = None

    while True:
        response = drive_service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken, files(id, name, mimeType)",
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
            pageSize=1000,
            pageToken=page_token,
        ).execute()

        results.extend(response.get("files", []))
        page_token = response.get("nextPageToken")

        if not page_token:
            break

    return results


def walk_drive_folder(root_folder_id):
    matched = []
    all_seen = []
    queue = deque([(root_folder_id, "")])

    while queue:
        folder_id, folder_path = queue.popleft()
        children = list_children(folder_id)

        for item in children:
            item_id = item["id"]
            item_name = item["name"]
            mime = item["mimeType"]
            current_path = f"{folder_path}/{item_name}" if folder_path else item_name

            if mime == "application/vnd.google-apps.folder":
                queue.append((item_id, current_path))
                all_seen.append({
                    "path": current_path,
                    "name": item_name,
                    "mimeType": mime,
                    "type": "folder",
                    "selected": False,
                    "document_type": None,
                    "reason": "folder",
                })
                continue

            selected = is_target_pdf_name(item_name, mime)
            document_type = detect_document_type(item_name)
            reason = explain_detection(item_name, mime)

            all_seen.append({
                "path": current_path,
                "name": item_name,
                "mimeType": mime,
                "type": "file",
                "selected": selected,
                "document_type": document_type if selected else None,
                "reason": reason,
            })

            if selected:
                matched.append({
                    "id": item_id,
                    "name": item_name,
                    "path": current_path,
                    "mimeType": mime,
                    "document_type": document_type,
                    "is_bch": document_type == "บัญชีรายชื่อ",
                })

    debug_df = pd.DataFrame(all_seen)
    debug_df.to_csv(DETECTION_DEBUG_OUTPUT, index=False, encoding="utf-8-sig")

    if DEBUG_FILE_DETECTION:
        print("Detection debug saved to:", DETECTION_DEBUG_OUTPUT)
        print("\n=== selected counts ===")
        print(debug_df["selected"].value_counts(dropna=False))
        print("\n=== document type counts among selected ===")
        print(pd.DataFrame(matched)["document_type"].value_counts(dropna=False) if matched else "No matched files")
        display(debug_df.head(50))

    return matched


def download_drive_file(file_id: str, out_path: Path):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    request = drive_service.files().get_media(
        fileId=file_id,
        supportsAllDrives=True,
    )

    fh = io.FileIO(str(out_path), "wb")
    downloader = MediaIoBaseDownload(fh, request)

    done = False
    while not done:
        status, done = downloader.next_chunk()
        if status:
            print(f"  Download {int(status.progress() * 100)}%")

    fh.close()
    return out_path


matched_files = walk_drive_folder(ROOT_FOLDER_ID)

print("Matched files:", len(matched_files))
for x in matched_files[:100]:
    print(f"- [{x['document_type']}] {x['path']}")

if MAX_FILES_TO_PROCESS is not None:
    matched_files = matched_files[:MAX_FILES_TO_PROCESS]
    print("Limited files:", len(matched_files))


downloaded = []

for i, item in enumerate(matched_files, start=1):
    print("=" * 80)
    print(f"[{i}/{len(matched_files)}] [{item['document_type']}] {item['path']}")

    subdir = DOWNLOAD_DIR / Path(item["path"]).parent
    subdir.mkdir(parents=True, exist_ok=True)

    out_path = subdir / safe_filename(item["name"])
    download_drive_file(item["id"], out_path)

    downloaded.append({
        "source_id": item["id"],
        "source_path": item["path"],
        "local_path": str(out_path),
        "document_type": item["document_type"],
        "is_bch": item["is_bch"],
    })

print("Downloaded files:", len(downloaded))
if downloaded:
    print(pd.DataFrame(downloaded)["document_type"].value_counts())


# ============================================================
# 6) Text / number helpers
# ============================================================

THAI_DIGIT_MAP = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")

THAI_NUM_WORDS = {
    "ศูนย์": 0,
    "หนึ่ง": 1,
    "เอ็ด": 1,
    "สอง": 2,
    "ยี่": 2,
    "สาม": 3,
    "สี่": 4,
    "ห้า": 5,
    "หก": 6,
    "เจ็ด": 7,
    "แปด": 8,
    "เก้า": 9,
}

THAI_UNITS = {
    "สิบ": 10,
    "ร้อย": 100,
    "พัน": 1000,
    "หมื่น": 10000,
    "แสน": 100000,
    "ล้าน": 1000000,
}


def normalize_text(x):
    if x is None:
        return None
    if isinstance(x, float) and math.isnan(x):
        return None
    s = str(x).translate(THAI_DIGIT_MAP)
    s = re.sub(r"\s+", " ", s).strip()
    if s == "" or s.lower() in ["nan", "none", "null"]:
        return None
    return s


def clean_int(value):
    value = normalize_text(value)
    if value is None:
        return None

    s = value.replace(",", "")
    m = re.search(r"-?\d+", s)
    return int(m.group()) if m else None


def thai_text_to_int(text):
    text = normalize_text(text)
    if text is None:
        return None

    digit_value = clean_int(text)
    if digit_value is not None:
        return digit_value

    s = text.replace(" ", "")
    total = 0
    current = 0
    i = 0

    while i < len(s):
        matched = False

        for word, val in sorted(THAI_NUM_WORDS.items(), key=lambda x: len(x[0]), reverse=True):
            if s.startswith(word, i):
                current = val
                i += len(word)
                matched = True
                break

        if matched:
            continue

        for unit_word, unit_val in sorted(THAI_UNITS.items(), key=lambda x: len(x[0]), reverse=True):
            if s.startswith(unit_word, i):
                if unit_word == "ล้าน":
                    if current == 0:
                        current = 1
                    total = (total + current) * unit_val
                    current = 0
                else:
                    if current == 0:
                        current = 1
                    total += current * unit_val
                    current = 0

                i += len(unit_word)
                matched = True
                break

        if matched:
            continue

        i += 1

    total += current
    return total if total != 0 else None


def resolve_dual_number(numeric_raw, text_raw, fallback=None, prefer_text_on_conflict=True):
    numeric_val = clean_int(numeric_raw)
    text_val = thai_text_to_int(text_raw)
    fallback_val = clean_int(fallback)

    if numeric_val is not None and text_val is not None:
        if numeric_val == text_val:
            final = numeric_val
            decision = "numeric_text_agree"
        else:
            final = text_val if prefer_text_on_conflict else numeric_val
            decision = "conflict_prefer_text" if prefer_text_on_conflict else "conflict_prefer_numeric"

    elif text_val is not None:
        final = text_val
        decision = "text_only"

    elif numeric_val is not None:
        final = numeric_val
        decision = "numeric_only"

    elif fallback_val is not None:
        final = fallback_val
        decision = "fallback"

    else:
        final = None
        decision = "missing"

    return {
        "final_value": final,
        "numeric_raw": normalize_text(numeric_raw),
        "text_raw": normalize_text(text_raw),
        "fallback_raw": normalize_text(fallback),
        "decision": decision,
    }


def fuzzy_pick(value, choices, cutoff=FUZZY_SCORE_CUTOFF):
    value = normalize_text(value)
    if value is None:
        return None, None
    if value in choices:
        return value, 100

    match = process.extractOne(value, choices, scorer=fuzz.WRatio)
    if match and match[1] >= cutoff:
        return match[0], match[1]

    return value, match[1] if match else None


def extract_metadata_from_source_path(source_path: str):
    s = str(source_path).replace("\\", "/")
    parts = [p for p in s.split("/") if p]

    amphoe = None
    tambon = None
    unit = None
    moo = None

    for p in parts:
        if p.startswith("อำเภอ") or p.startswith("ทต."):
            amphoe = p
        if p.startswith("ตำบล") or p.startswith("ทต."):
            tambon = p

        m_unit = re.search(r"หน่วย(?:เลือกตั้ง)?(?:ที่)?\s*([0-9]+)", p)
        if m_unit:
            unit = m_unit.group(1)

        m_moo = re.search(r"หมู่(?:ที่)?\s*([0-9]+)", p)
        if m_moo:
            moo = m_moo.group(1)

    return {
        "หน่วยเลือกตั้งที่": unit,
        "หมู่ที่": moo,
        "ตำบล/แขวง/เทศบาล": tambon,
        "อำเภอ/เขต": amphoe,
    }


def ensure_columns(df, columns):
    df = df.copy()
    for col in columns:
        if col not in df.columns:
            df[col] = None
    return df[columns]


# ============================================================
# 7) JSON repair helper
# ============================================================

def extract_json_text(text: str) -> str:
    text = (text or "").strip()
    text = re.sub(r"^```json\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1 and end > start:
        text = text[start:end + 1]

    return text.strip()


def repair_json_text(text: str) -> str:
    text = extract_json_text(text)

    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("‘", "'").replace("’", "'")

    text = re.sub(r",\s*([}\]])", r"\1", text)

    text = re.sub(
        r'([{\[,]\s*)([A-Za-z0-9_\u0E00-\u0E7F\/\(\)\- ]+?)\s*:',
        lambda m: f'{m.group(1)}"{m.group(2).strip()}":',
        text
    )

    return text


def safe_json_loads(text: str, debug_path: Path = None):
    raw = text or ""

    try:
        return json.loads(extract_json_text(raw))
    except Exception:
        pass

    repaired = repair_json_text(raw)

    try:
        return json.loads(repaired)
    except Exception:
        pass

    try:
        return ast.literal_eval(repaired)
    except Exception:
        if debug_path is not None:
            debug_path = Path(debug_path)
            debug_path.parent.mkdir(parents=True, exist_ok=True)
            debug_path.with_suffix(".raw.txt").write_text(raw, encoding="utf-8")
            debug_path.with_suffix(".repaired.txt").write_text(repaired, encoding="utf-8")

        raise ValueError("Cannot parse Gemini JSON response")


# ============================================================
# 8) PDF to images + OCR
# ============================================================

def pdf_to_images(pdf_path: Path, out_dir: Path, zoom: float = PDF_ZOOM):
    pdf_path = Path(pdf_path)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    doc = fitz.open(str(pdf_path))
    img_paths = []
    mat = fitz.Matrix(zoom, zoom)

    for page_index in range(len(doc)):
        page = doc[page_index]
        pix = page.get_pixmap(matrix=mat, alpha=False)

        out_path = out_dir / f"page_{page_index + 1:03d}.png"
        pix.save(str(out_path))
        img_paths.append(out_path)

    doc.close()
    return img_paths


def gemini_generate_with_retry(model, contents, config=None, label="gemini"):
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            if REQUEST_DELAY_SECONDS:
                time.sleep(REQUEST_DELAY_SECONDS)

            return client.models.generate_content(
                model=model,
                contents=contents,
                config=config,
            )

        except Exception as e:
            last_error = e
            wait = min(
                INITIAL_BACKOFF_SECONDS * (BACKOFF_MULTIPLIER ** (attempt - 1)),
                MAX_BACKOFF_SECONDS
            )
            wait = wait + random.uniform(0, 2)
            print(f"WARNING: {label} attempt {attempt}/{MAX_RETRIES} failed: {e}")
            print(f"Retrying in {wait:.1f}s...")
            time.sleep(wait)

    raise RuntimeError(f"{label} failed after {MAX_RETRIES} attempts: {last_error}")


OCR_PROMPT = """
You are an OCR engine for Thai election forms.

Read the image carefully and return markdown text only.

Important:
- Preserve Thai text.
- Preserve table structure as much as possible.
- Keep candidate numbers, party numbers, names, party names, vote numbers, and Thai number words.
- Do not summarize.
- Do not translate.
- If a value is handwritten, still extract it as accurately as possible.
"""


def ocr_image_to_markdown(img_path: Path) -> str:
    img_path = Path(img_path)
    data = img_path.read_bytes()

    resp = gemini_generate_with_retry(
        model=OCR_MODEL,
        contents=[
            OCR_PROMPT,
            types.Part.from_bytes(
                data=data,
                mime_type="image/png"
            )
        ],
        config=types.GenerateContentConfig(
            temperature=0.0,
            max_output_tokens=10000,
        ),
        label=f"ocr:{img_path.name}",
    )

    return (resp.text or "").strip()


def run_ocr_on_images(img_paths: list, work_dir: Path):
    work_dir = Path(work_dir)
    md_dir = work_dir / "markdown_pages"
    md_dir.mkdir(parents=True, exist_ok=True)

    page_markdowns = []

    for i, img_path in enumerate(img_paths, start=1):
        print(f"OCR page {i}/{len(img_paths)}: {img_path.name}")

        md = ocr_image_to_markdown(img_path)

        md_path = md_dir / f"page_{i:03d}.md"
        md_path.write_text(md, encoding="utf-8")

        page_markdowns.append({
            "page": i,
            "image_path": str(img_path),
            "markdown": md,
        })

    full_md = "\n\n".join([
        f"\n\n<!-- PAGE {x['page']} -->\n\n{x['markdown']}"
        for x in page_markdowns
    ])

    (work_dir / "full_ocr.md").write_text(full_md, encoding="utf-8")

    return page_markdowns, full_md


# ============================================================
# 9) Parser
# ============================================================

CONSTITUENCY_SYSTEM_PROMPT = """
You are an expert Thai election constituency result form parser.

Document type is "แบ่งเขต".

Extract strict JSON only.

Rules:
- Every JSON object key must be enclosed in double quotes.
- Do not use trailing commas.
- Do not use comments inside JSON.
- Return valid JSON only.
- If a value is unclear, return null.
- Preserve Thai text exactly when possible.
- Normalize visible digits to Arabic numerals.
- Extract only rows visible in this OCR page/chunk.
- Candidate rows should include candidate number, candidate name, party affiliation, votes.
- If the row is "ไม่เลือกผู้ใด", put candidate name as "ไม่เลือกผู้ใด" and party as null.
- Summary values such as บัตรดี, บัตรเสีย, บัตรที่ไม่เลือก should be in summary.

Output schema:
{
  "summary": {
    "ประเภทเอกสาร": "แบ่งเขต",
    "หน่วยเลือกตั้งที่": null,
    "หน่วยเลือกตั้งที่(ตัวอักษร)": null,
    "หมู่ที่": null,
    "หมู่ที่(ตัวอักษร)": null,
    "ตำบล/แขวง/เทศบาล": null,
    "อำเภอ/เขต": null,
    "จำนวนผู้มีสิทธิ": null,
    "จำนวนผู้มาใช้สิทธิ์": null,
    "บัตรดี": null,
    "บัตรเสีย": null,
    "บัตรที่ไม่เลือก": null
  },
  "candidates": [
    {
      "หมายเลขผู้สมัคร": null,
      "ชื่อผู้สมัคร": null,
      "สังกัดพรรคการเมือง": null,
      "จำนวนคะแนนที่ได้": null,
      "จำนวนที่ได้(ตัวอักษร)": null
    }
  ]
}
"""


PARTYLIST_SYSTEM_PROMPT = """
You are an expert Thai election party-list result form parser.

Document type is "บัญชีรายชื่อ".

Extract strict JSON only.

Rules:
- Every JSON object key must be enclosed in double quotes.
- Do not use trailing commas.
- Do not use comments inside JSON.
- Return valid JSON only.
- If a value is unclear, return null.
- Preserve Thai text exactly when possible.
- Normalize visible digits to Arabic numerals.
- Extract only rows visible in this OCR page/chunk.
- There is NO candidate name.
- There is NO candidate affiliation.
- Extract party-list rows with party number, party name, votes.
- If the row is "ไม่เลือกผู้ใด", use เลขพรรค=null and ชื่อพรรค="ไม่เลือกผู้ใด".
- Summary values such as บัตรดี, บัตรเสีย, บัตรที่ไม่เลือก should be in summary.

Output schema:
{
  "summary": {
    "ประเภทเอกสาร": "บัญชีรายชื่อ",
    "หน่วยเลือกตั้งที่": null,
    "หน่วยเลือกตั้งที่(ตัวอักษร)": null,
    "หมู่ที่": null,
    "หมู่ที่(ตัวอักษร)": null,
    "ตำบล/แขวง/เทศบาล": null,
    "อำเภอ/เขต": null,
    "จำนวนผู้มีสิทธิ": null,
    "จำนวนผู้มาใช้สิทธิ์": null,
    "บัตรดี": null,
    "บัตรเสีย": null,
    "บัตรที่ไม่เลือก": null
  },
  "party_results": [
    {
      "เลขพรรค": null,
      "ชื่อพรรค": null,
      "จำนวนคะแนนที่ได้": null,
      "จำนวนที่ได้(ตัวอักษร)": null
    }
  ]
}
"""


def parse_markdown_to_json(
    ocr_markdown: str,
    source_path: str,
    document_type: str,
    page_label: str = "full",
):
    meta = extract_metadata_from_source_path(source_path)

    if document_type == "บัญชีรายชื่อ":
        system_prompt = PARTYLIST_SYSTEM_PROMPT
        task_hint = """
This is a party-list form.
Do NOT extract candidate names.
Extract only party number, party name, and votes.
"""
    else:
        system_prompt = CONSTITUENCY_SYSTEM_PROMPT
        task_hint = """
This is a constituency form.
Extract candidate number, candidate name, party affiliation, and votes.
"""

    user_prompt = f"""
Extract election data from the OCR markdown below.

Known context from file path:
- source_pdf: {source_path}
- document_type: {document_type}
- known หน่วยเลือกตั้งที่: {meta['หน่วยเลือกตั้งที่']}
- known หมู่ที่: {meta['หมู่ที่']}
- known ตำบล/แขวง/เทศบาล: {meta['ตำบล/แขวง/เทศบาล']}
- known อำเภอ/เขต: {meta['อำเภอ/เขต']}
- valid อำเภอ/เขต choices: {LOCATION_LEXICON['อำเภอ/เขต']}
- valid ตำบล/แขวง/เทศบาล choices: {LOCATION_LEXICON['ตำบล/แขวง/เทศบาล']}

{task_hint}

Important:
- This may be only one page of the document.
- Extract rows visible on this page only.
- Use null for fields not visible.
- For vote scores, extract both visible digit score and Thai number words if present.
- Return strict valid JSON only.

OCR markdown:
```markdown
{ocr_markdown}
```
"""

    resp = gemini_generate_with_retry(
        model=PARSER_MODEL,
        contents=[system_prompt, user_prompt],
        config=types.GenerateContentConfig(
            temperature=0.0,
            max_output_tokens=10000,
            response_mime_type="application/json",
        ),
        label=f"parse:{document_type}:{Path(source_path).name}:{page_label}",
    )

    content = (resp.text or "").strip()

    debug_json_path = (
        OCR_DIR
        / safe_filename(Path(source_path).stem)
        / f"gemini_parse_response_{safe_filename(page_label)}.json"
    )

    return safe_json_loads(content, debug_path=debug_json_path)


# ============================================================
# 10) Postprocess
# ============================================================

def clean_summary(parsed_summary: dict, source_path: str, document_type: str):
    meta = extract_metadata_from_source_path(source_path)
    summary = parsed_summary or {}

    unit_info = resolve_dual_number(
        summary.get("หน่วยเลือกตั้งที่"),
        summary.get("หน่วยเลือกตั้งที่(ตัวอักษร)"),
        fallback=meta["หน่วยเลือกตั้งที่"],
        prefer_text_on_conflict=True,
    )

    moo_info = resolve_dual_number(
        summary.get("หมู่ที่"),
        summary.get("หมู่ที่(ตัวอักษร)"),
        fallback=meta["หมู่ที่"],
        prefer_text_on_conflict=True,
    )

    tambon_raw = meta["ตำบล/แขวง/เทศบาล"] or summary.get("ตำบล/แขวง/เทศบาล")
    amphoe_raw = meta["อำเภอ/เขต"] or summary.get("อำเภอ/เขต")

    tambon_fixed, tambon_score = fuzzy_pick(
        tambon_raw,
        LOCATION_LEXICON["ตำบล/แขวง/เทศบาล"],
    )

    amphoe_fixed, amphoe_score = fuzzy_pick(
        amphoe_raw,
        LOCATION_LEXICON["อำเภอ/เขต"],
    )

    return {
        "source_pdf": source_path,
        "ประเภทเอกสาร": document_type,
        "is_bch": document_type == "บัญชีรายชื่อ",

        "หน่วยเลือกตั้งที่": unit_info["final_value"],
        "หน่วยเลือกตั้งที่(ตัวเลขดิบ)": unit_info["numeric_raw"],
        "หน่วยเลือกตั้งที่(ตัวอักษร)": unit_info["text_raw"],
        "หน่วยเลือกตั้งที่_from_path": meta["หน่วยเลือกตั้งที่"],
        "หน่วยเลือกตั้งที่_decision": unit_info["decision"],

        "หมู่ที่": moo_info["final_value"],
        "หมู่ที่(ตัวเลขดิบ)": moo_info["numeric_raw"],
        "หมู่ที่(ตัวอักษร)": moo_info["text_raw"],
        "หมู่ที่_from_path": meta["หมู่ที่"],
        "หมู่ที่_decision": moo_info["decision"],

        "ตำบล/แขวง/เทศบาล": tambon_fixed,
        "อำเภอ/เขต": amphoe_fixed,

        "จำนวนผู้มีสิทธิ": clean_int(summary.get("จำนวนผู้มีสิทธิ")),
        "จำนวนผู้มาใช้สิทธิ์": clean_int(summary.get("จำนวนผู้มาใช้สิทธิ์")),
        "บัตรดี": clean_int(summary.get("บัตรดี")),
        "บัตรเสีย": clean_int(summary.get("บัตรเสีย")),
        "บัตรไม่เลือกผู้ใด": clean_int(
            summary.get("บัตรไม่เลือกผู้ใด")
            or summary.get("บัตรที่ไม่เลือก")
            or summary.get("บัตรที่ไม่เลือกผู้ใด")
        ),

        "match_score_ตำบล": tambon_score,
        "match_score_อำเภอ": amphoe_score,
    }


def merge_summary(best_summary, new_summary):
    if best_summary is None:
        return dict(new_summary)

    for k, v in new_summary.items():
        if best_summary.get(k) is None and v is not None:
            best_summary[k] = v

    return best_summary


def postprocess_constituency_page(parsed: dict, source_path: str):
    summary = clean_summary(
        parsed.get("summary", {}) or {},
        source_path,
        "แบ่งเขต",
    )

    rows = []

    for cand in parsed.get("candidates", []) or []:
        score_info = resolve_dual_number(
            cand.get("จำนวนคะแนนที่ได้"),
            cand.get("จำนวนที่ได้(ตัวอักษร)"),
            fallback=None,
            prefer_text_on_conflict=True,
        )

        candidate_no = clean_int(cand.get("หมายเลขผู้สมัคร"))
        candidate_name = normalize_text(cand.get("ชื่อผู้สมัคร"))
        party = normalize_text(cand.get("สังกัดพรรคการเมือง"))

        if candidate_name and "ไม่เลือก" in candidate_name:
            candidate_no = None
            candidate_name = "ไม่เลือกผู้ใด"
            party = None

        if candidate_no is None and candidate_name is None and party is None and score_info["final_value"] is None:
            continue

        rows.append({
            **summary,
            "หมายเลขผู้สมัคร": candidate_no,
            "ชื่อผู้สมัคร": candidate_name,
            "สังกัดพรรคการเมือง": party,
            "จำนวนคะแนนที่ได้": score_info["final_value"],
            "จำนวนคะแนนที่ได้(ตัวเลขดิบ)": score_info["numeric_raw"],
            "จำนวนที่ได้(ตัวอักษร)": score_info["text_raw"],
            "จำนวนคะแนนที่ได้_decision": score_info["decision"],
        })

    return summary, rows


def postprocess_partylist_page(parsed: dict, source_path: str):
    summary = clean_summary(
        parsed.get("summary", {}) or {},
        source_path,
        "บัญชีรายชื่อ",
    )

    party_results = (
        parsed.get("party_results")
        or parsed.get("candidates")
        or parsed.get("rows")
        or []
    )

    rows = []

    for row in party_results:
        party_no = clean_int(row.get("เลขพรรค"))
        party_name = (
            normalize_text(row.get("ชื่อพรรค"))
            or normalize_text(row.get("พรรค"))
            or normalize_text(row.get("party_name"))
            or normalize_text(row.get("party"))
            or normalize_text(row.get("ชื่อผู้สมัคร"))
            or normalize_text(row.get("สังกัดพรรคการเมือง"))
        )

        if party_name and "ไม่เลือก" in party_name:
            party_no = None
            party_name = "ไม่เลือกผู้ใด"

        score_info = resolve_dual_number(
            row.get("จำนวนคะแนนที่ได้"),
            row.get("จำนวนที่ได้(ตัวอักษร)"),
            fallback=None,
            prefer_text_on_conflict=True,
        )

        if party_no is None and party_name is None and score_info["final_value"] is None:
            continue

        rows.append({
            **summary,
            "เลขพรรค": party_no,
            "ชื่อพรรค": party_name,
            "จำนวนคะแนนที่ได้": score_info["final_value"],
            "จำนวนคะแนนที่ได้(ตัวเลขดิบ)": score_info["numeric_raw"],
            "จำนวนที่ได้(ตัวอักษร)": score_info["text_raw"],
            "จำนวนคะแนนที่ได้_decision": score_info["decision"],
        })

    return summary, rows


def fill_summary_into_rows(rows, summary):
    final = []
    for row in rows:
        row = dict(row)
        for col in [
            "source_pdf", "ประเภทเอกสาร", "is_bch",
            "หน่วยเลือกตั้งที่", "หมู่ที่",
            "ตำบล/แขวง/เทศบาล", "อำเภอ/เขต",
            "จำนวนผู้มีสิทธิ", "จำนวนผู้มาใช้สิทธิ์",
            "บัตรดี", "บัตรเสีย", "บัตรไม่เลือกผู้ใด",
            "match_score_ตำบล", "match_score_อำเภอ",
        ]:
            if row.get(col) is None and summary.get(col) is not None:
                row[col] = summary.get(col)

        final.append(row)

    return final


def dedupe_rows(rows, key_cols):
    out = []
    seen = set()

    for row in rows:
        key = tuple(row.get(c) for c in key_cols)
        if key in seen:
            continue
        seen.add(key)
        out.append(row)

    return out


# ============================================================
# 11) Main processing
# ============================================================

summary_rows = []
constituency_rows = []
partylist_rows = []
errors = []

print("downloaded files to process =", len(downloaded))

for item in tqdm(downloaded):
    local_pdf = Path(item["local_path"])
    rel_name = item["source_path"]
    document_type = item["document_type"]

    try:
        work_dir = OCR_DIR / safe_filename(local_pdf.stem)
        img_paths = pdf_to_images(local_pdf, work_dir / "images", zoom=PDF_ZOOM)

        print("=" * 100)
        print(f"PROCESSING [{document_type}]: {rel_name}")
        print("num_pages =", len(img_paths))

        pages, full_md = run_ocr_on_images(img_paths, work_dir)
        print("ocr_chars =", len(full_md))

        all_page_parsed_raw = []
        all_rows = []
        best_summary = None

        # Parse ทีละหน้าเสมอ เพื่อลดปัญหา JSON พังจาก OCR ยาวเกิน
        for page_obj in pages:
            page_no = page_obj["page"]
            page_md = page_obj["markdown"]

            if not page_md or len(page_md.strip()) < 20:
                continue

            print(f"Parsing page {page_no}/{len(pages)}")

            try:
                parsed_raw_page = parse_markdown_to_json(
                    page_md,
                    rel_name,
                    document_type=document_type,
                    page_label=f"page_{page_no:03d}",
                )

                all_page_parsed_raw.append({
                    "page": page_no,
                    "parsed_raw": parsed_raw_page,
                })

                if document_type == "บัญชีรายชื่อ":
                    page_summary, page_rows = postprocess_partylist_page(
                        parsed_raw_page,
                        rel_name,
                    )
                else:
                    page_summary, page_rows = postprocess_constituency_page(
                        parsed_raw_page,
                        rel_name,
                    )

                best_summary = merge_summary(best_summary, page_summary)
                all_rows.extend(page_rows)

            except Exception as page_error:
                print(f"PAGE PARSE ERROR page {page_no}: {page_error}")

                errors.append({
                    "source_pdf": rel_name,
                    "page": page_no,
                    "ประเภทเอกสาร": document_type,
                    "is_bch": document_type == "บัญชีรายชื่อ",
                    "error": str(page_error),
                })

        (work_dir / "parsed_raw_pages.json").write_text(
            json.dumps(all_page_parsed_raw, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

        if best_summary is None:
            raise ValueError("No page could be parsed successfully")

        final_rows = fill_summary_into_rows(all_rows, best_summary)

        if document_type == "บัญชีรายชื่อ":
            final_rows = dedupe_rows(
                final_rows,
                ["source_pdf", "เลขพรรค", "ชื่อพรรค", "จำนวนคะแนนที่ได้"],
            )
            partylist_rows.extend(final_rows)
        else:
            final_rows = dedupe_rows(
                final_rows,
                ["source_pdf", "หมายเลขผู้สมัคร", "ชื่อผู้สมัคร", "จำนวนคะแนนที่ได้"],
            )
            constituency_rows.extend(final_rows)

        summary_rows.append(best_summary)

        parsed_clean = {
            "summary": best_summary,
            "rows": final_rows,
        }

        (work_dir / "parsed_clean.json").write_text(
            json.dumps(parsed_clean, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

        print(f"Extracted rows: {len(final_rows)}")

    except Exception as e:
        print("ERROR:", rel_name, e)
        errors.append({
            "source_pdf": rel_name,
            "page": None,
            "ประเภทเอกสาร": document_type,
            "is_bch": document_type == "บัญชีรายชื่อ",
            "error": str(e),
        })


summary_df = pd.DataFrame(summary_rows)
constituency_df = pd.DataFrame(constituency_rows)
partylist_df = pd.DataFrame(partylist_rows)
errors_df = pd.DataFrame(errors)

print("\nDone.")
print("Summary rows:", len(summary_df))
print("Constituency rows:", len(constituency_df))
print("Party-list rows:", len(partylist_df))
print("Errors:", len(errors_df))


# ============================================================
# 12) Export
# ============================================================

CONSTITUENCY_COLUMNS = [
    "source_pdf",
    "หน่วยเลือกตั้งที่",
    "หมู่ที่",
    "ตำบล/แขวง/เทศบาล",
    "อำเภอ/เขต",
    "จำนวนผู้มีสิทธิ",
    "จำนวนผู้มาใช้สิทธิ์",
    "บัตรดี",
    "บัตรเสีย",
    "บัตรไม่เลือกผู้ใด",
    "หมายเลขผู้สมัคร",
    "ชื่อผู้สมัคร",
    "สังกัดพรรคการเมือง",
    "จำนวนคะแนนที่ได้",
    "จำนวนคะแนนที่ได้(ตัวเลขดิบ)",
    "จำนวนที่ได้(ตัวอักษร)",
    "จำนวนคะแนนที่ได้_decision",
]

PARTYLIST_COLUMNS = [
    "source_pdf",
    "หน่วยเลือกตั้งที่",
    "หมู่ที่",
    "ตำบล/แขวง/เทศบาล",
    "อำเภอ/เขต",
    "บัตรดี",
    "บัตรเสีย",
    "บัตรไม่เลือกผู้ใด",
    "เลขพรรค",
    "ชื่อพรรค",
    "จำนวนคะแนนที่ได้",
    "จำนวนคะแนนที่ได้(ตัวเลขดิบ)",
    "จำนวนที่ได้(ตัวอักษร)",
    "จำนวนคะแนนที่ได้_decision",
]

constituency_out = ensure_columns(constituency_df, CONSTITUENCY_COLUMNS)
partylist_out = ensure_columns(partylist_df, PARTYLIST_COLUMNS)

constituency_out.to_csv(CONSTITUENCY_OUTPUT, index=False, encoding="utf-8-sig")
partylist_out.to_csv(PARTYLIST_OUTPUT, index=False, encoding="utf-8-sig")
summary_df.to_csv(SUMMARY_OUTPUT, index=False, encoding="utf-8-sig")
errors_df.to_csv(ERROR_OUTPUT, index=False, encoding="utf-8-sig")

print("Exported main files:")
print("1)", CONSTITUENCY_OUTPUT)
print("2)", PARTYLIST_OUTPUT)

print("\nExtra debug files:")
print("3)", SUMMARY_OUTPUT)
print("4)", ERROR_OUTPUT)
print("5)", DETECTION_DEBUG_OUTPUT)

print("\nShapes:")
print("constituency_out:", constituency_out.shape)
print("partylist_out:", partylist_out.shape)
print("summary_df:", summary_df.shape)
print("errors_df:", errors_df.shape)

display(constituency_out.head(20))
display(partylist_out.head(20))
display(errors_df.head(20))


Google Drive auth completed.
Gemini client created.
OCR_MODEL = gemini-2.5-flash
PARSER_MODEL = gemini-2.5-flash
ROOT_FOLDER_ID = 170C8kzr9J6VzdykufsLGWAckjtdHeoxA


Detection debug saved to: /content/file_detection_debug.csv

=== selected counts ===
selected
True     573
False    329
Name: count, dtype: int64

=== document type counts among selected ===
document_type
บัญชีรายชื่อ    288
แบ่งเขต         285
Name: count, dtype: int64


,path,name,mimeType,type,selected,document_type,reason
0,อำเภอไชยปราการ,อำเภอไชยปราการ,application/vnd.google-apps.folder,folder,False,None,folder
1,อำเภอแม่อาย,อำเภอแม่อาย,application/vnd.google-apps.folder,folder,False,None,folder
2,ทต.แม่อาย,ทต.แม่อาย,application/vnd.google-apps.folder,folder,False,None,folder
3,ล่วงหน้านอกเขตและนอกราชอาณาจักร,ล่วงหน้านอกเขตและนอกราชอาณาจักร,application/vnd.google-apps.folder,folder,False,None,folder
4,อำเภอฝาง,อำเภอฝาง,application/vnd.google-apps.folder,folder,False,None,folder
5,อำเภอไชยปราการ/ตำบลแม่ทะลบ,ตำบลแม่ทะลบ,application/vnd.google-apps.folder,folder,False,None,folder
6,อำเภอไชยปราการ/ตำบลปงตำ,ตำบลปงตำ,application/vnd.google-apps.folder,folder,False,None,folder
7,อำเภอแม่อาย/ตำบลแม่สาว,ตำบลแม่สาว,application/vnd.google-apps.folder,folder,False,None,folder
8,อำเภอแม่อาย/ตำบลแม่อาย,ตำบลแม่อาย,application/vnd.google-apps.folder,folder,False,None,folder
9,อำเภอแม่อาย/ตำบลแม่นาวาง,ตำบลแม่นาวาง,application/vnd.google-apps.folder,folder,False,None,folder


Matched files: 573
- [บัญชีรายชื่อ] อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 9/สส. 5ทับ18 (บช).pdf
- [แบ่งเขต] อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 9/สส. 5ทับ18.pdf
- [บัญชีรายชื่อ] อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 6/สส. 5ทับ18 (บช).pdf
- [แบ่งเขต] อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 6/สส. 5ทับ18.pdf
- [บัญชีรายชื่อ] อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 8/สส. 5ทับ18 (บช).pdf
- [แบ่งเขต] อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 8/สส.5ทับ18.pdf
- [แบ่งเขต] อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 5/สส. 5ทับ18.pdf
- [บัญชีรายชื่อ] อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 5/สส. 5ทับ18 (บช).pdf
- [บัญชีรายชื่อ] อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 7/สส. 5ทับ18 (บช).pdf
- [แบ่งเขต] อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 7/สส.5ทับ18.pdf
- [บัญชีรายชื่อ] อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 4/สส. 5ทับ18 (บช).pdf
- [แบ่งเขต] อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 4/สส. 5ทับ18.pdf
- [บัญชีรายชื่อ] อำเภอไชยปราการ/ตำบ

  0%|          | 0/573 [00:00<?, ?it/s]

เอาต์พุตของการสตรีมมีการตัดเหลือเพียง 5000 บรรทัดสุดท้าย
Parsing page 1/2
Parsing page 2/2
Extracted rows: 9
PROCESSING [บัญชีรายชื่อ]: อำเภอแม่อาย/ตำบลแม่นาวาง/หน่วยเลือกตั้งที่ 1/ส.ส.5 ทับ 18 บช. หน่วย 1.pdf
num_pages = 4
OCR page 1/4: page_001.png
OCR page 2/4: page_002.png
OCR page 3/4: page_003.png
OCR page 4/4: page_004.png
ocr_chars = 10108
Parsing page 1/4
Parsing page 2/4
Parsing page 3/4
Parsing page 4/4
Extracted rows: 57
PROCESSING [แบ่งเขต]: อำเภอแม่อาย/ตำบลแม่นาวาง/หน่วยเลือกตั้งที่ 1/สส. 5ทับ18.pdf
num_pages = 2
OCR page 1/2: page_001.png
OCR page 2/2: page_002.png
ocr_chars = 134184
Parsing page 1/2
Parsing page 2/2
Extracted rows: 9
PROCESSING [บัญชีรายชื่อ]: อำเภอแม่อาย/ตำบลบ้านหลวง/หน่วยเลือกตั้งที่ 9/สส.5ทับ18 บช หน่วยที่ 9.pdf
num_pages = 4
OCR page 1/4: page_001.png
OCR page 2/4: page_002.png
OCR page 3/4: page_003.png
OCR page 4/4: page_004.png
ocr_chars = 148932
Parsing page 1/4
Parsing page 2/4
Parsing page 3/4
Parsing page 4/4
Extracted rows: 57
PROCESSING [แบ

,source_pdf,หน่วยเลือกตั้งที่,หมู่ที่,ตำบล/แขวง/เทศบาล,อำเภอ/เขต,จำนวนผู้มีสิทธิ,จำนวนผู้มาใช้สิทธิ์,บัตรดี,บัตรเสีย,บัตรไม่เลือกผู้ใด,หมายเลขผู้สมัคร,ชื่อผู้สมัคร,สังกัดพรรคการเมือง,จำนวนคะแนนที่ได้,จำนวนคะแนนที่ได้(ตัวเลขดิบ),จำนวนที่ได้(ตัวอักษร),จำนวนคะแนนที่ได้_decision
0,อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 9...,1,7.0,ตำบลแม่ทะลบ,อำเภอไชยปราการ,559.0,395.0,389.0,6.0,NaN,1.0,นายอดุลย์ บุญใส,ภูมิใจไทย,0.0,0,ศูนย์,numeric_only
1,อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 9...,1,7.0,ตำบลแม่ทะลบ,อำเภอไชยปราการ,559.0,395.0,389.0,6.0,NaN,2.0,นายสมดุลย์ อุดเจริญ,ประชาชน,31.0,31,สามสิบเอ็ดบัตร,numeric_text_agree
2,อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 9...,1,7.0,ตำบลแม่ทะลบ,อำเภอไชยปราการ,559.0,395.0,389.0,6.0,NaN,3.0,นายเกรียงกมล ศรีมา,ประชาธิปัตย์,1.0,1,หนึ่งบัตร,numeric_text_agree
3,อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 9...,1,7.0,ตำบลแม่ทะลบ,อำเภอไชยปราการ,559.0,395.0,389.0,6.0,NaN,4.0,นายสันติ ตันสุขี,ก้าวอิสระ,1.0,1,หนึ่งบัตร,numeric_text_agree
4,อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 9...,1,7.0,ตำบลแม่ทะลบ,อำเภอไชยปราการ,559.0,395.0,389.0,6.0,NaN,5.0,นายการุณย์ คูเจริญชัยกุล,กล้าธรรม,318.0,318,สามร้อยสิบแปดบัตร,numeric_text_agree
5,อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 9...,1,7.0,ตำบลแม่ทะลบ,อำเภอไชยปราการ,559.0,395.0,389.0,6.0,NaN,6.0,นายนิธิกร วุฒินันชัย,เพื่อไทย,30.0,30,สามสิบบัตร,numeric_text_agree
6,อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 9...,1,7.0,ตำบลแม่ทะลบ,อำเภอไชยปราการ,559.0,395.0,389.0,6.0,NaN,7.0,นายไกร ดาบธรรม,พลังประชารัฐ,7.0,7,เจ็ดบัตร,numeric_text_agree
7,อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 9...,1,7.0,ตำบลแม่ทะลบ,อำเภอไชยปราการ,559.0,395.0,389.0,6.0,NaN,8.0,นายพิสัณห์ อุปนันท์,เศรษฐกิจ,1.0,1,หนึ่งบัตร,numeric_text_agree
8,อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 9...,1,7.0,ตำบลแม่ทะลบ,อำเภอไชยปราการ,559.0,395.0,389.0,6.0,NaN,9.0,นายชาญวิทยายุทธ์ อินทร์แก้ว,ประชากรไทย,0.0,0,ศูนย์,numeric_only
9,อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 6...,6,5.0,ตำบลแม่ทะลบ,อำเภอไชยปราการ,533.0,402.0,378.0,10.0,14.0,1.0,นายอดุลย์ บุญใส,ภูมิใจไทย,12.0,12,สิบสองคะแนน,numeric_text_agree


,source_pdf,หน่วยเลือกตั้งที่,หมู่ที่,ตำบล/แขวง/เทศบาล,อำเภอ/เขต,บัตรดี,บัตรเสีย,บัตรไม่เลือกผู้ใด,เลขพรรค,ชื่อพรรค,จำนวนคะแนนที่ได้,จำนวนคะแนนที่ได้(ตัวเลขดิบ),จำนวนที่ได้(ตัวอักษร),จำนวนคะแนนที่ได้_decision
0,อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 9...,9,NaN,ตำบลแม่ทะลบ,อำเภอไชยปราการ,389.0,NaN,NaN,11.0,เศรษฐกิจ,0.0,0,ศูนย์,numeric_only
1,อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 9...,9,NaN,ตำบลแม่ทะลบ,อำเภอไชยปราการ,389.0,NaN,NaN,12.0,เสรีรวมไทย,0.0,0,ศูนย์,numeric_only
2,อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 9...,9,NaN,ตำบลแม่ทะลบ,อำเภอไชยปราการ,389.0,NaN,NaN,13.0,รวมพลังประชาชน,0.0,0,ศูนย์,numeric_only
3,อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 9...,9,NaN,ตำบลแม่ทะลบ,อำเภอไชยปราการ,389.0,NaN,NaN,14.0,ท้องที่ไทย,1.0,1,หนึ่ง,numeric_text_agree
4,อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 9...,9,NaN,ตำบลแม่ทะลบ,อำเภอไชยปราการ,389.0,NaN,NaN,15.0,อนาคตไทย,0.0,0,ศูนย์,numeric_only
5,อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 9...,9,NaN,ตำบลแม่ทะลบ,อำเภอไชยปราการ,389.0,NaN,NaN,16.0,พลังเพื่อไทย,3.0,3,สาม,numeric_text_agree
6,อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 9...,9,NaN,ตำบลแม่ทะลบ,อำเภอไชยปราการ,389.0,NaN,NaN,17.0,ไทยชนะ,0.0,0,ศูนย์,numeric_only
7,อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 9...,9,NaN,ตำบลแม่ทะลบ,อำเภอไชยปราการ,389.0,NaN,NaN,18.0,พลังสังคมใหม่,1.0,1,หนึ่ง,numeric_text_agree
8,อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 9...,9,NaN,ตำบลแม่ทะลบ,อำเภอไชยปราการ,389.0,NaN,NaN,19.0,สังคมประชาธิปไตยไทย,0.0,0,ศูนย์,numeric_only
9,อำเภอไชยปราการ/ตำบลแม่ทะลบ/หน่วยเลือกตั้งที่ 9...,9,NaN,ตำบลแม่ทะลบ,อำเภอไชยปราการ,389.0,NaN,NaN,20.0,ฟิวชั่น,0.0,0,ศูนย์,numeric_only


,source_pdf,page,ประเภทเอกสาร,is_bch,error
0,ทต.แม่อาย/ตำบลมะลิกา/หน่วยเลือกตั้งที่ 18/สส. ...,1.0,แบ่งเขต,False,Cannot parse Gemini JSON response
1,ทต.แม่อาย/ตำบลมะลิกา/หน่วยเลือกตั้งที่ 17/สส. ...,2.0,บัญชีรายชื่อ,True,Cannot parse Gemini JSON response
2,อำเภอฝาง/ตำบลแม่สูน/หน่วยเลือกตั้งที่ 9/สส. 5 ...,3.0,บัญชีรายชื่อ,True,Cannot parse Gemini JSON response
3,อำเภอฝาง/ตำบลแม่สูน/หน่วยเลือกตั้งที่ 15/สส. 5...,3.0,บัญชีรายชื่อ,True,Cannot parse Gemini JSON response
4,อำเภอฝาง/ตำบลแม่งอน/หน่วยเลือกตั้งที่ 3/ส.ส. 5...,4.0,บัญชีรายชื่อ,True,Cannot parse Gemini JSON response
5,อำเภอฝาง/ทต.เวียงฝาง/หน่วยเลือกตั้งที่ 6/สส. 5...,3.0,บัญชีรายชื่อ,True,Cannot parse Gemini JSON response
6,อำเภอฝาง/ตำบลแม่คะ/หน่วยเลือกตั้งที่ 4/สส. 5ทั...,3.0,บัญชีรายชื่อ,True,Cannot parse Gemini JSON response
7,อำเภอฝาง/ตำบลม่อนปิ่น/หน่วยเลือกตั้งที่ 20/สส....,3.0,บัญชีรายชื่อ,True,Cannot parse Gemini JSON response
8,อำเภอฝาง/ตำบลม่อนปิ่น/หน่วยเลือกตั้งที่ 19/สส....,1.0,แบ่งเขต,False,Cannot parse Gemini JSON response
9,อำเภอฝาง/ตำบลม่อนปิ่น/หน่วยเลือกตั้งที่ 19/สส....,2.0,แบ่งเขต,False,parse:แบ่งเขต:สส. 5ทับ18.pdf:page_002 failed a...


## ไฟล์ผลลัพธ์

หลังรันจบ ให้ดาวน์โหลดไฟล์จาก Colab:

1. `/content/constituency_results_gemini.csv`
2. `/content/partylist_results_gemini.csv`

ไฟล์ช่วยตรวจ:
- `/content/errors_gemini.csv`
- `/content/file_detection_debug.csv`
- `/content/all_summary_gemini.csv`
